In [ ]:
from typesafe_sdk import TypeSafeClient, Noul, Score
print("Jev SDK imported successfully")

In [ ]:
import os
import time
import json
from getpass import getpass
from pathlib import Path

import numpy as np
import pandas as pd

from typesafe_sdk import TypeSafeClient, Noul, Score

if not os.getenv("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass("TypeSafe API key: ")

client = TypeSafeClient()

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "train.csv").exists():
    ROOT = ROOT.parent

train = pd.read_csv(ROOT / "train.csv")

TARGETS = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

print(train.shape)
train.head()

In [ ]:
import sys

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from jev_labeling import generate_labels

ARTIFACTS = ROOT / "artifacts"
OUTPUT_CSV = ARTIFACTS / "train_jev_choice_labeled.csv"
seed_labels = pd.read_csv(ROOT / "report_labels_v2.csv")
LABEL_LIMIT = None  # Run every observation; use an integer for a smaller test.
AUDIT_ALL = True  # Ask JEV about every target for comparison with v2.

labeled_train, label_errors = generate_labels(
    train,
    client,
    checkpoint_path=ARTIFACTS / "jev_choice_checkpoint.jsonl",
    output_path=OUTPUT_CSV,
    seed_labels=seed_labels,
    limit=LABEL_LIMIT,
    workers=4,
    audit_all=AUDIT_ALL,
)

print(f"Errors: {len(label_errors)}")
print(f"Saved labeled data to {OUTPUT_CSV}")
labeled_train.head()

In [ ]:
coverage = pd.DataFrame({
    target: labeled_train[f"{target}__verdict"].value_counts()
    for target in TARGETS
}).T.fillna(0).astype(int)
display(coverage.reindex(columns=["YES", "NO", "UNK"], fill_value=0))

jev_columns = [f"{target}__jev_verdict" for target in TARGETS]
v2_columns = [f"{target}__verdict" for target in TARGETS]
comparison = labeled_train[["StudyInstanceUID", "Report", *jev_columns]].merge(
    seed_labels[["StudyInstanceUID", *v2_columns]],
    on="StudyInstanceUID",
    how="inner",
)

summary = []
conflicts = []
for target in TARGETS:
    jev = comparison[f"{target}__jev_verdict"]
    v2 = comparison[f"{target}__verdict"]
    evaluated = jev.notna()
    comparable = evaluated & jev.isin(["YES", "NO"]) & v2.isin(["YES", "NO"])
    agreed = comparable & jev.eq(v2)
    conflict = comparable & jev.ne(v2)
    total = int(comparable.sum())
    summary.append({
        "target": target,
        "jev_evaluated": int(evaluated.sum()),
        "comparable_yes_no": total,
        "agreements": int(agreed.sum()),
        "conflicts": int(conflict.sum()),
        "agreement_rate": agreed.sum() / total if total else np.nan,
        "v2_unk": int((evaluated & v2.eq("UNK")).sum()),
        "jev_unk": int((evaluated & jev.eq("UNK")).sum()),
    })
    for _, row in comparison.loc[conflict].iterrows():
        conflicts.append({
            "StudyInstanceUID": row["StudyInstanceUID"],
            "target": target,
            "JEV": row[f"{target}__jev_verdict"],
            "report_labels_v2": row[f"{target}__verdict"],
            "report": " ".join(str(row["Report"]).split())[:240],
        })

comparison_summary = pd.DataFrame(summary).set_index("target")
display(comparison_summary)
print(
    f"Overall decisive agreement: {comparison_summary['agreements'].sum()}/"
    f"{comparison_summary['comparable_yes_no'].sum()}"
)
display(pd.DataFrame(conflicts))

In [ ]:
metrics = []
decisive_evaluations = 0
decisive_agreements = 0

for target in TARGETS:
    jev = comparison[f"{target}__jev_verdict"]
    v2 = comparison[f"{target}__verdict"]
    evaluated = jev.notna() & v2.notna()
    agreed = evaluated & jev.eq(v2)
    conflict = evaluated & jev.ne(v2)
    count = int(evaluated.sum())
    agreement_count = int(agreed.sum())
    metrics.append({
        "Target": target,
        "# Evaluations": count,
        "# Agreements": agreement_count,
        "# Conflicts": int(conflict.sum()),
        "% Match": agreement_count / count if count else np.nan,
    })

    decisive = evaluated & jev.isin(["YES", "NO"]) & v2.isin(["YES", "NO"])
    decisive_evaluations += int(decisive.sum())
    decisive_agreements += int((decisive & jev.eq(v2)).sum())

metrics_table = pd.DataFrame(metrics).set_index("Target")
total_evaluations = int(metrics_table["# Evaluations"].sum())
total_agreements = int(metrics_table["# Agreements"].sum())
metrics_table.loc["Total"] = {
    "# Evaluations": total_evaluations,
    "# Agreements": total_agreements,
    "# Conflicts": int(metrics_table["# Conflicts"].sum()),
    "% Match": total_agreements / total_evaluations if total_evaluations else np.nan,
}

display(metrics_table.style.format({
    "# Evaluations": "{:.0f}",
    "# Agreements": "{:.0f}",
    "# Conflicts": "{:.0f}",
    "% Match": "{:.1%}",
}))
METRICS_CSV = ARTIFACTS / "jev_v2_comparison_metrics.csv"
metrics_table.reset_index().to_csv(METRICS_CSV, index=False)
print(f"Saved comparison metrics to {METRICS_CSV}")
print(
    f"Decisive-only match: {decisive_agreements}/{decisive_evaluations} "
    f"({decisive_agreements / decisive_evaluations:.1%})"
    if decisive_evaluations else "No decisive YES/NO comparisons available."
)